# DAPT-01 QLoRA resume 3893 → 5500

Qwen3-8B-Base, 4-bit QLoRA, locked DAPT-01 config.

- Save every 500 steps → upload each checkpoint to Kaggle
- Status/log heartbeat → `Priyanshu47/dapt01-train-status` (pollable without any local VM)
- This notebook is run/monitored by the agent; you can just open it to watch

In [ ]:
# @title 1) Environment + GPU check
import os, sys, subprocess, json
from pathlib import Path

print("python", sys.version)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "bitsandbytes>=0.46.1", "datasets", "peft", "transformers",
                "accelerate", "kaggle"], check=False)
import torch, bitsandbytes, transformers, peft, datasets
print("deps", bitsandbytes.__version__, transformers.__version__, peft.__version__, torch.__version__)
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

In [ ]:
# @title 2) Download domain train + checkpoint-3893 from Kaggle
import os, json, shutil, subprocess, sys
from pathlib import Path

token = os.environ.get("KAGGLE_API_TOKEN") or os.environ.get("KAGGLE_KEY")
user = os.environ.get("KAGGLE_USERNAME", "Priyanshu47")
assert token, "KAGGLE_API_TOKEN / KAGGLE_KEY not set in runtime env"
cfg = Path.home() / ".kaggle"
cfg.mkdir(exist_ok=True)
(cfg / "kaggle.json").write_text(json.dumps({"username": user, "key": token}))
os.chmod(cfg / "kaggle.json", 0o600)
env = os.environ.copy()
env["KAGGLE_CONFIG_DIR"] = str(cfg)

content = Path("/content")
dt = content / "domain_train"
if not (dt / "train.jsonl").exists():
    subprocess.run(["kaggle", "datasets", "download", "-d", "priyanshu47/jainllm-domain-train",
                    "-p", str(dt), "--unzip"], env=env, check=True)
else:
    print("domain_train already present")

ck = content / "checkpoint-3893"
if not (ck / "trainer_state.json").exists():
    dl = content / "ck_dl"
    subprocess.run(["kaggle", "datasets", "download", "-d", "priyanshu47/dapt01-checkpoint-3893",
                    "-p", str(dl), "--unzip"], env=env, check=True)
    cands = list(dl.rglob("adapter_config.json"))
    src = cands[0].parent if cands else dl
    if ck.exists():
        shutil.rmtree(ck)
    ck.mkdir()
    for p in src.iterdir():
        if p.is_file():
            shutil.copy2(p, ck / p.name)
else:
    print("checkpoint-3893 already present")

ts = json.loads((ck / "trainer_state.json").read_text())
assert ts["global_step"] == 3893, ts["global_step"]
print("READY step", ts["global_step"], "train_bytes", (dt / "train.jsonl").stat().st_size)

In [ ]:
# @title 3) Write runtime env (Kaggle token for ckpt upload + status push)
import os, json
from pathlib import Path

token = os.environ.get("KAGGLE_API_TOKEN") or os.environ.get("KAGGLE_KEY")
user = os.environ.get("KAGGLE_USERNAME", "Priyanshu47")
assert token
Path("/content/dapt_train_env.json").write_text(json.dumps({
    "KAGGLE_API_TOKEN": token,
    "KAGGLE_KEY": token,
    "KAGGLE_USERNAME": user,
    "DAPT_TARGET_STEP": "5500",
}, indent=2))
os.chmod("/content/dapt_train_env.json", 0o600)
os.environ["DAPT_TARGET_STEP"] = "5500"
print("env written; target=5500")

In [ ]:
# @title 4) Launch detached training + Kaggle status pusher (returns immediately)
import subprocess, sys
from pathlib import Path

for name in ["dapt_train_worker.py", "relaunch_worker.py", "status_pusher.py", "nb_launch.py", "train_status.py"]:
    p = Path("/content") / name
    assert p.exists(), f"missing {name} — upload scripts first"

r = subprocess.run([sys.executable, "-u", "/content/nb_launch.py"],
                   capture_output=True, text=True, timeout=180)
print(r.stdout)
if r.stderr:
    print(r.stderr, file=sys.stderr)
assert r.returncode == 0, f"launch failed rc={r.returncode}"
assert "LAUNCH_OK" in r.stdout
print("NB_LAUNCH_OK")

In [ ]:
# @title 5) Status snapshot (safe to re-run anytime)
import subprocess, sys
r = subprocess.run([sys.executable, "-u", "/content/train_status.py"],
                   capture_output=True, text=True, timeout=60)
print(r.stdout)
if r.stderr:
    print(r.stderr, file=sys.stderr)

## Poll without this notebook / any local VM

```bash
kaggle datasets download -d Priyanshu47/dapt01-train-status -p /tmp/dapt_status --unzip
cat /tmp/dapt_status/status_compact.txt
```

Checkpoints appear as:
`Priyanshu47/dapt01-checkpoint-4000` (then 4500, 5000, 5500).